In [2]:
import os 
import sys
import warnings
import subprocess
import pandas as pd
import numpy as np
import pybedtools
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
from gene2probe import *

This tutorial guides you through the design of custom probes against a gene of interest.

First, we need to specify the gene symbol, and which feature we are interested in designing probes agains (in this case we are using exons).
We also need to specify an output directory for the analysis.

### 1. Specify parameters

In [3]:
## Specify gene of interest and feature of interest
gene_ID = 'CDKN2B-AS1'
mode = 'transcript' ## Whether   to consider only exons / introns or full gene

## specify output directory
out_dir = '../sample_run/probeDesign_' + gene_ID + '_' + mode + '/'
## Create output directory
os.makedirs(out_dir, exist_ok=True)

Additionally, we need to provide the path to several resource files. Many of these files can be obtained from [UCSC table browser](https://genome.ucsc.edu/cgi-bin/hgTables).

We also need a blast database, such as the one we generated in the [previous tutorial](https://github.com/Teichlab/gene2probe/blob/main/notebooks/001_make_blast_database.ipynb).

In [4]:
## Required resources (most can be downloaded from 
gtf = '../hg38_resources/hg38.ncbiRefSeq.gtf' ## Gene annotation in gtf file
## We recommend using RefSeq as this is manually curated and more likely to contain an isoform that is present across most cell types
## Alternatively, one can filter based on RNA-seq data for a cell type/tissue of interest
fasta = '../hg38_resources/hg38.fa' ## Genome in fasta file
snp_db = '../hg38_resources/hg38_snp151Common.bed' ## Database of known SNPs and small indels
repeats = '../hg38_resources/hg38_rmsk.bed' ## bed file with repeats/low complexity regions to be excluded
gaps = '../hg38_resources/hg38_rmsk.bed' ## bed file with gaps in the genome assembly to be excluded
blast_db = '../hg38_resources/001_blastdb/hg38_ncbiRefSeq_transcripts_db' ## Database of all human transcripts to blast against

In [5]:
## Path to blast binaries.
## Replace with your conda environment
## This can also be omitted if you started the jupyter session from within the gene2probe conda environment
print('current working directory:', os.getcwd())

blast_exec_path = f"{os.environ['HOME']}/.miniforge3/envs/gene2probe_env/bin/"

if not os.path.isdir(blast_exec_path):
    warnings.warn(
        f'BLAST executable directory not found: {blast_exec_path}',
        RuntimeWarning,
    )

print('blast executable path:', blast_exec_path)

current working directory: /Users/dangriffiths/gene2probe/notebooks
blast executable path: /Users/dangriffiths/.miniforge3/envs/gene2probe_env/bin/


Finally, we need to provide a set of parameters related to our probe's length, at which nucleotide it's split (if at all), the acceptable range for GC content and any specific requirements for individual nucleotides.

Here we are following the [recommendations of 10x Genomics for custom probes for VisiumHD/VisiumFFPE/Flex](https://cdn.10xgenomics.com/image/upload/v1697739385/support-documents/CG000621_CustomProbeDesign_TechNote_RevC.pdf).

In [6]:
## Additional parameters regarding how the probe should look like
probe_length = 50 ## Length of probe in nucleotides
split_nt = 25 ## Index of nucleotide to split the probe at (start of RHS) - set to None if splitting probe is not needed
min_GC = 0.44 ## Minimum GC content for probe (if split probe, applied to both LHS and RHS)
max_GC = 0.72 ## Maximum GC content for probe (if split probe, applied to both LHS and RHS)
required_nts = {24: 'T'} ## Dictionary of index (0-based) for required nts - by default, 25th nucleotide must be a T - set to None if no requirements
probe_offset = 1000 ## Minimum distance between probes - 10 bp is the recommended minimum by 10x, this can also be adjusted depending on how many probes pass other cutoffs
n_desired_probes =3 ## Number of probes to be designed.
min_mismatches = 5 ## Minimum number of mismatches (in at least LHS or RHS) - here we require in both to be more conservative

In [7]:
## Optionally, we can also specify adapters that have to be added to the probes.
## For example, for visiumHD:
LHS_pref = 'CCTTGGCACCCGAGAATTCCA' ## Will be added to the 5' of the LHS probe
LHS_suff = '' ## Will be added to the 3' of the LHS probe
RHS_pref = '/5Phos/' ## Will be added to the 5' of the RHS probe
RHS_suff = 'CCCATATAAGAAA' ## Will be added to the 3' of the RHS probe

## Leave as empty strings if you don't want to use them


### 2. Generate k-mers

Now we can start by reading the gene annotation and filtering for our gene of interest.

In [8]:
## Read gtf file
gene_anno = read_gtf(gtf)

In [9]:
gene_anno

,seqname,source,feature,start,end,score,strand,frame,attribute
0,chrM,ncbiRefSeq.2022-10-28,transcript,15956,16023,.,-,.,"gene_id ""TRNP""; transcript_id ""rna-TRNP""; gen..."
1,chrM,ncbiRefSeq.2022-10-28,exon,15956,16023,.,-,.,"gene_id ""TRNP""; transcript_id ""rna-TRNP""; exon..."
2,chrM,ncbiRefSeq.2022-10-28,transcript,15888,15953,.,+,.,"gene_id ""TRNT""; transcript_id ""rna-TRNT""; gen..."
3,chrM,ncbiRefSeq.2022-10-28,exon,15888,15953,.,+,.,"gene_id ""TRNT""; transcript_id ""rna-TRNT""; exon..."
4,chrM,ncbiRefSeq.2022-10-28,transcript,14747,15887,.,+,.,"gene_id ""CYTB""; transcript_id ""rna-CYTB""; gen..."
...,...,...,...,...,...,...,...,...,...
4886697,chr1,ncbiRefSeq.2022-10-28,exon,29321,29370,.,-,.,"gene_id ""WASH7P""; transcript_id ""NR_024540.1"";..."
4886698,chr1,ncbiRefSeq.2022-10-28,transcript,11874,14409,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."
4886699,chr1,ncbiRefSeq.2022-10-28,exon,11874,12227,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."
4886700,chr1,ncbiRefSeq.2022-10-28,exon,12613,12721,.,+,.,"gene_id ""DDX11L1""; transcript_id ""NR_046018.2""..."


In [10]:
## Extract regions corresponding to gene of interest (symbol: gene_name, Ensembl ID: gene_ID), subset to feature of interest and convert to bed style dataframe:
roi_bed = get_region_of_interest(gene_anno, gene_ID, gene_id_type = 'gene_name', feature=mode)

In [11]:
roi_bed

,seqname,start,end,name,score,strand
0,chr9,21994790,22077890,CDKN2B-AS1_0,.,+
1,chr9,21994790,22077890,CDKN2B-AS1_1,.,+
2,chr9,21994790,22077890,CDKN2B-AS1_2,.,+
3,chr9,21994790,22077890,CDKN2B-AS1_3,.,+
4,chr9,21994790,22077890,CDKN2B-AS1_4,.,+
5,chr9,21994790,22121097,CDKN2B-AS1_5,.,+
6,chr9,21994790,22121097,CDKN2B-AS1_6,.,+
7,chr9,21994790,22121097,CDKN2B-AS1_7,.,+
8,chr9,21994790,22121097,CDKN2B-AS1_8,.,+
9,chr9,21994790,22121097,CDKN2B-AS1_9,.,+


After extracting the coordinates of interest and converting to a bed-like format, we can generate all possible kmers that fall within these regions.

In [12]:
kmers = generate_kmers(roi_bed, k=probe_length)

In [13]:
kmers

,seqname,start,end,name,score,strand
0,chr9,21994790,21994840,CDKN2B-AS1_0_0,.,+
1,chr9,21994791,21994841,CDKN2B-AS1_0_1,.,+
2,chr9,21994792,21994842,CDKN2B-AS1_0_2,.,+
3,chr9,21994793,21994843,CDKN2B-AS1_0_3,.,+
4,chr9,21994794,21994844,CDKN2B-AS1_0_4,.,+
...,...,...,...,...,...,...
1551572,chr9,22121043,22121093,CDKN2B-AS1_13_126253,.,+
1551573,chr9,22121044,22121094,CDKN2B-AS1_13_126254,.,+
1551574,chr9,22121045,22121095,CDKN2B-AS1_13_126255,.,+
1551575,chr9,22121046,22121096,CDKN2B-AS1_13_126256,.,+


In [14]:
## Export unfiltered
kmers.to_csv((out_dir + 'kmers_all.csv'))

### 3. Exclude annotated repeats/polymorphism

We can next exclude kmers overlapping undesired regions (repeats, low complexity regions, common polymorphism, gaps in the assembly) from further consideration.

Ideally, we will exclude everything that overlaps a repeat or polymorphism, but if we only have too few kmers available, we might need to relax these requirements (e.g., to only exclude kmers overlapping SNPs around the ligation junction, if the probes are split).

In [15]:
## For example, we could have removed all kmers overlapping a repeat/low complexity region within 5 nts of the ligation junction:
remove_overlaps(kmers, repeats, core=[20,30])

,seqname,start,end,name,score,strand
0,chr9,21994790,21994840,CDKN2B-AS1_0_0,.,+
1,chr9,21994791,21994841,CDKN2B-AS1_0_1,.,+
2,chr9,21994792,21994842,CDKN2B-AS1_0_2,.,+
3,chr9,21994793,21994843,CDKN2B-AS1_0_3,.,+
4,chr9,21994794,21994844,CDKN2B-AS1_0_4,.,+
...,...,...,...,...,...,...
833401,chr9,22121043,22121093,CDKN2B-AS1_13_126253,.,+
833402,chr9,22121044,22121094,CDKN2B-AS1_13_126254,.,+
833403,chr9,22121045,22121095,CDKN2B-AS1_13_126255,.,+
833404,chr9,22121046,22121096,CDKN2B-AS1_13_126256,.,+


In [16]:
## In this case we have a lot of possible kmers, so we will remove those with overlaps in any part of the probe:
kmers = remove_overlaps(kmers, repeats)

In [17]:
kmers 

,seqname,start,end,name,score,strand
0,chr9,21994790,21994840,CDKN2B-AS1_0_0,.,+
1,chr9,21994791,21994841,CDKN2B-AS1_0_1,.,+
2,chr9,21994792,21994842,CDKN2B-AS1_0_2,.,+
3,chr9,21994793,21994843,CDKN2B-AS1_0_3,.,+
4,chr9,21994794,21994844,CDKN2B-AS1_0_4,.,+
...,...,...,...,...,...,...
798761,chr9,22120695,22120745,CDKN2B-AS1_13_125905,.,+
798762,chr9,22120696,22120746,CDKN2B-AS1_13_125906,.,+
798763,chr9,22120697,22120747,CDKN2B-AS1_13_125907,.,+
798764,chr9,22120698,22120748,CDKN2B-AS1_13_125908,.,+


In [18]:
## Doing the same for gaps in the assembly (very unlikely since we are starting with annotated exons)
kmers = remove_overlaps(kmers, gaps)

In [19]:
## And more importantly, against common polymorphism (SNPs, short indels)
kmers = remove_overlaps(kmers, snp_db)

In [20]:
kmers

,seqname,start,end,name,score,strand
0,chr9,21994790,21994840,CDKN2B-AS1_0_0,.,+
1,chr9,21994791,21994841,CDKN2B-AS1_0_1,.,+
2,chr9,21994792,21994842,CDKN2B-AS1_0_2,.,+
3,chr9,21994793,21994843,CDKN2B-AS1_0_3,.,+
4,chr9,21994794,21994844,CDKN2B-AS1_0_4,.,+
...,...,...,...,...,...,...
657448,chr9,22120695,22120745,CDKN2B-AS1_13_125905,.,+
657449,chr9,22120696,22120746,CDKN2B-AS1_13_125906,.,+
657450,chr9,22120697,22120747,CDKN2B-AS1_13_125907,.,+
657451,chr9,22120698,22120748,CDKN2B-AS1_13_125908,.,+


### 4. Filter for desirable sequence features

Having excluded undesirable kmers based on intersection with genomic annotations, the next step is to consider their sequence features.

For this, we first extract the sequences of each k-mer, and then estimate features such as GC content and the presence of desired nucleotides in specific positions.

In [21]:
## Get DNA for the transcript
kmers_bed = pybedtools.BedTool.from_dataframe(kmers)
kmers_seq = kmers_bed.sequence(fi=fasta, s=True) 

## We can read in the sequences and simultaneously monitor GC content and count the longest homopolymer stretch
kmers_seq_stats = get_sequence_stats(kmers_seq.seqfn, probe_length, split_nt)

In [22]:
## Combining with our dataframe
kmers = pd.merge(kmers, kmers_seq_stats, left_index=True, right_index=True)

In [23]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS
0,chr9,21994790,21994840,CDKN2B-AS1_0_0,.,+,chr9:21994790-21994840(+),AGCTACATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCT...,CGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGACGGATGT...,0.66,3,0.72,0.60
1,chr9,21994791,21994841,CDKN2B-AS1_0_1,.,+,chr9:21994791-21994841(+),GCTACATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTC...,GCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGACGGATG...,0.68,3,0.72,0.64
2,chr9,21994792,21994842,CDKN2B-AS1_0_2,.,+,chr9:21994792-21994842(+),CTACATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCC...,CGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGACGGAT...,0.68,3,0.72,0.64
3,chr9,21994793,21994843,CDKN2B-AS1_0_3,.,+,chr9:21994793-21994843(+),TACATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCC...,CCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGACGGA...,0.68,3,0.76,0.60
4,chr9,21994794,21994844,CDKN2B-AS1_0_4,.,+,chr9:21994794-21994844(+),ACATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCG...,TCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGACGG...,0.68,3,0.76,0.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...
657448,chr9,22120695,22120745,CDKN2B-AS1_13_125905,.,+,chr9:22120695-22120745(+),TTATCCAGTAATAACATATTGGCATATGTCTTTCTGGTATATTTTC...,ACAGGAAAATATACCAGAAAGACATATGCCAATATGTTATTACTGG...,0.30,4,0.32,0.28
657449,chr9,22120696,22120746,CDKN2B-AS1_13_125906,.,+,chr9:22120696-22120746(+),TATCCAGTAATAACATATTGGCATATGTCTTTCTGGTATATTTTCC...,CACAGGAAAATATACCAGAAAGACATATGCCAATATGTTATTACTG...,0.32,4,0.36,0.28
657450,chr9,22120697,22120747,CDKN2B-AS1_13_125907,.,+,chr9:22120697-22120747(+),ATCCAGTAATAACATATTGGCATATGTCTTTCTGGTATATTTTCCT...,ACACAGGAAAATATACCAGAAAGACATATGCCAATATGTTATTACT...,0.32,4,0.36,0.28
657451,chr9,22120698,22120748,CDKN2B-AS1_13_125908,.,+,chr9:22120698-22120748(+),TCCAGTAATAACATATTGGCATATGTCTTTCTGGTATATTTTCCTG...,AACACAGGAAAATATACCAGAAAGACATATGCCAATATGTTATTAC...,0.32,4,0.32,0.32


In [24]:
## Check for required nucleotides in specific positions:
if required_nts is not None:
    kmers['has_required_nts'] = check_for_required_nts(kmers, required_nts)
    print(kmers['has_required_nts'].value_counts())
    ## Filter for required nucleotides
    kmers = kmers[kmers['has_required_nts']==True].reset_index(drop=True)

has_required_nts
False    461295
True     196158
Name: count, dtype: int64


In [25]:
## Export kmers before filtering
kmers.to_csv((out_dir + 'kmers_candidates_unfiltered.csv'))

In [26]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr9,21994793,21994843,CDKN2B-AS1_0_3,.,+,chr9:21994793-21994843(+),TACATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCC...,CCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGACGGA...,0.68,3,0.76,0.60,True
1,chr9,21994796,21994846,CDKN2B-AS1_0_6,.,+,chr9:21994796-21994846(+),ATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCGCG...,AATCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGAC...,0.66,3,0.68,0.64,True
2,chr9,21994799,21994849,CDKN2B-AS1_0_9,.,+,chr9:21994799-21994849(+),CGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCGCGGAT...,CAGAATCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGT...,0.68,3,0.68,0.68,True
3,chr9,21994800,21994850,CDKN2B-AS1_0_10,.,+,chr9:21994800-21994850(+),GTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCGCGGATT...,CCAGAATCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGG...,0.68,3,0.72,0.64,True
4,chr9,21994802,21994852,CDKN2B-AS1_0_12,.,+,chr9:21994802-21994852(+),CACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCGCGGATTCT...,CACCAGAATCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCA...,0.68,3,0.72,0.64,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196153,chr9,22120683,22120733,CDKN2B-AS1_13_125893,.,+,chr9:22120683-22120733(+),AGAAAATAAAAATTATCCAGTAATAACATATTGGCATATGTCTTTC...,ACCAGAAAGACATATGCCAATATGTTATTACTGGATAATTTTTATT...,0.26,5,0.36,0.16,True
196154,chr9,22120685,22120735,CDKN2B-AS1_13_125895,.,+,chr9:22120685-22120735(+),AAAATAAAAATTATCCAGTAATAACATATTGGCATATGTCTTTCTG...,ATACCAGAAAGACATATGCCAATATGTTATTACTGGATAATTTTTA...,0.24,5,0.32,0.16,True
196155,chr9,22120687,22120737,CDKN2B-AS1_13_125897,.,+,chr9:22120687-22120737(+),AATAAAAATTATCCAGTAATAACATATTGGCATATGTCTTTCTGGT...,ATATACCAGAAAGACATATGCCAATATGTTATTACTGGATAATTTT...,0.24,5,0.32,0.16,True
196156,chr9,22120693,22120743,CDKN2B-AS1_13_125903,.,+,chr9:22120693-22120743(+),AATTATCCAGTAATAACATATTGGCATATGTCTTTCTGGTATATTT...,AGGAAAATATACCAGAAAGACATATGCCAATATGTTATTACTGGAT...,0.28,4,0.28,0.28,True


In [27]:
## Filter for GC content
kmers = filter_by_GC_content(kmers, min_GC, max_GC)

In [28]:
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr9,21994796,21994846,CDKN2B-AS1_0_6,.,+,chr9:21994796-21994846(+),ATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCGCG...,AATCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGAC...,0.66,3,0.68,0.64,True
1,chr9,21994799,21994849,CDKN2B-AS1_0_9,.,+,chr9:21994799-21994849(+),CGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCGCGGAT...,CAGAATCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGT...,0.68,3,0.68,0.68,True
2,chr9,21994858,21994908,CDKN2B-AS1_0_68,.,+,chr9:21994858-21994908(+),GCGTCCCCGCTCCCCTATTCCCCTTATTTTATTCCTGGCTCCCCTC...,CGACGAGGGGAGCCAGGAATAAAATAAGGGGAATAGGGGAGCGGGG...,0.60,4,0.52,0.68,True
3,chr9,21994863,21994913,CDKN2B-AS1_0_73,.,+,chr9:21994863-21994913(+),CCCGCTCCCCTATTCCCCTTATTTTATTCCTGGCTCCCCTCGTCGA...,ACTTTCGACGAGGGGAGCCAGGAATAAAATAAGGGGAATAGGGGAG...,0.54,4,0.56,0.52,True
4,chr9,21994952,21995002,CDKN2B-AS1_0_162,.,+,chr9:21994952-21995002(+),GAAGAAAGGAAAGCGAGGTCATCTCATTGCTCTATCCGCCAATCAG...,CCTCCTGATTGGCGGATAGAGCAATGAGATGACCTCGCTTTCCTTT...,0.50,3,0.52,0.48,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11767,chr9,22120231,22120281,CDKN2B-AS1_13_125441,.,+,chr9:22120231-22120281(+),ATGCATGAGCTATTGAGGCCTTTGCAGCTTTCTGCTACATGGAGGC...,CCTAGCCTCCATGTAGCAGAAAGCTGCAAAGGCCTCAATAGCTCAT...,0.50,3,0.52,0.48,True
11768,chr9,22120242,22120292,CDKN2B-AS1_13_125452,.,+,chr9:22120242-22120292(+),ATTGAGGCCTTTGCAGCTTTCTGCTACATGGAGGCTAGGGCCAGAG...,TTGACTCTGGCCCTAGCCTCCATGTAGCAGAAAGCTGCAAAGGCCT...,0.52,3,0.56,0.48,True
11769,chr9,22120244,22120294,CDKN2B-AS1_13_125454,.,+,chr9:22120244-22120294(+),TGAGGCCTTTGCAGCTTTCTGCTACATGGAGGCTAGGGCCAGAGTC...,TCTTGACTCTGGCCCTAGCCTCCATGTAGCAGAAAGCTGCAAAGGC...,0.54,3,0.56,0.52,True
11770,chr9,22120248,22120298,CDKN2B-AS1_13_125458,.,+,chr9:22120248-22120298(+),GCCTTTGCAGCTTTCTGCTACATGGAGGCTAGGGCCAGAGTCAAGA...,TAAATCTTGACTCTGGCCCTAGCCTCCATGTAGCAGAAAGCTGCAA...,0.50,3,0.48,0.52,True


In [29]:
## Candidate kmers
kmers

,seqname,start,end,name,score,strand,kmer_coord,transcript_seq,probe_seq,GC_content_full,longest_homopolymer,GC_content_LHS,GC_content_RHS,has_required_nts
0,chr9,21994796,21994846,CDKN2B-AS1_0_6,.,+,chr9:21994796-21994846(+),ATCCGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCGCG...,AATCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGTGAC...,0.66,3,0.68,0.64,True
1,chr9,21994799,21994849,CDKN2B-AS1_0_9,.,+,chr9:21994799-21994849(+),CGTCACCTGACACGGCCCTACCAGGAACAGCCGCGCTCCCGCGGAT...,CAGAATCCGCGGGAGCGCGGCTGTTCCTGGTAGGGCCGTGTCAGGT...,0.68,3,0.68,0.68,True
2,chr9,21994858,21994908,CDKN2B-AS1_0_68,.,+,chr9:21994858-21994908(+),GCGTCCCCGCTCCCCTATTCCCCTTATTTTATTCCTGGCTCCCCTC...,CGACGAGGGGAGCCAGGAATAAAATAAGGGGAATAGGGGAGCGGGG...,0.60,4,0.52,0.68,True
3,chr9,21994863,21994913,CDKN2B-AS1_0_73,.,+,chr9:21994863-21994913(+),CCCGCTCCCCTATTCCCCTTATTTTATTCCTGGCTCCCCTCGTCGA...,ACTTTCGACGAGGGGAGCCAGGAATAAAATAAGGGGAATAGGGGAG...,0.54,4,0.56,0.52,True
4,chr9,21994952,21995002,CDKN2B-AS1_0_162,.,+,chr9:21994952-21995002(+),GAAGAAAGGAAAGCGAGGTCATCTCATTGCTCTATCCGCCAATCAG...,CCTCCTGATTGGCGGATAGAGCAATGAGATGACCTCGCTTTCCTTT...,0.50,3,0.52,0.48,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11767,chr9,22120231,22120281,CDKN2B-AS1_13_125441,.,+,chr9:22120231-22120281(+),ATGCATGAGCTATTGAGGCCTTTGCAGCTTTCTGCTACATGGAGGC...,CCTAGCCTCCATGTAGCAGAAAGCTGCAAAGGCCTCAATAGCTCAT...,0.50,3,0.52,0.48,True
11768,chr9,22120242,22120292,CDKN2B-AS1_13_125452,.,+,chr9:22120242-22120292(+),ATTGAGGCCTTTGCAGCTTTCTGCTACATGGAGGCTAGGGCCAGAG...,TTGACTCTGGCCCTAGCCTCCATGTAGCAGAAAGCTGCAAAGGCCT...,0.52,3,0.56,0.48,True
11769,chr9,22120244,22120294,CDKN2B-AS1_13_125454,.,+,chr9:22120244-22120294(+),TGAGGCCTTTGCAGCTTTCTGCTACATGGAGGCTAGGGCCAGAGTC...,TCTTGACTCTGGCCCTAGCCTCCATGTAGCAGAAAGCTGCAAAGGC...,0.54,3,0.56,0.52,True
11770,chr9,22120248,22120298,CDKN2B-AS1_13_125458,.,+,chr9:22120248-22120298(+),GCCTTTGCAGCTTTCTGCTACATGGAGGCTAGGGCCAGAGTCAAGA...,TAAATCTTGACTCTGGCCCTAGCCTCCATGTAGCAGAAAGCTGCAA...,0.50,3,0.48,0.52,True


In [30]:
kmers.to_csv((out_dir + 'kmers_candidates_filtered.csv'))

### 5. Remove probes with potential off-targets

Having identified a set of kmers that fulfill our sequence requirements, we can next proceed with testing whether they are specific to our transcript/exon of interest.

For this we rely on using BLAST. 

We recommend blasting against transcripts (i.e., exons and introns combined) to be as conservative as possible in terms of off-targets. 
However, in cases where it is not possible to obtain enough suitable kmers (e.g., for  short transcripts), it is reasonable to relax this requirement by BLASTing against exons only (a much smaller search space).

At this step, we also want to consider whether our probes are split (as in the current specifications for VisiumHD) or a single oligo. If probes are split, it's best to BLAST each side separately, to make sure that both sides are specific.

In [31]:
## The first thing to do is to export our sequences in fasta format, so that we can use them for BLAST
write_fasta(kmers['name'], kmers['transcript_seq'], (out_dir + 'kmers_candidates_filtered_transcript_seqs.fa'))
## If our probes are meant to be split, we should additionally blast them separately 
## Note that the LHS/RHS in the transcript are reversed compared to the probe (i.e., the LHS of the transcript is complementary to the RHS of the probe)
if split_nt is not None: 
    ## Make split probes
    kmers['transcript_seq_LHS'] = [seq[0:split_nt] for seq in kmers['transcript_seq']]
    kmers['transcript_seq_RHS'] = [seq[split_nt: probe_length] for seq in kmers['transcript_seq']]

    ## We are exporting the transcript sequence as that's the one that has to be blasted against the human transcriptome
    write_fasta(kmers['name'], kmers['transcript_seq_LHS'], (out_dir + 'kmers_candidates_filtered_transcript_seqs_LHS.fa'))
    write_fasta(kmers['name'], kmers['transcript_seq_RHS'], (out_dir + 'kmers_candidates_filtered_transcript_seqs_RHS.fa'))    

In [ ]:
blast_res = {}
## First, blast the full probe
blast_res['full'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs.fa'),
                              blastdb = blast_db,
                              path2blastn=(blast_exec_path + 'blastn'),
                              outfile = (out_dir + 'kmers_candidates_filtered_blast_output.txt'))

## Additionally, if probe is split, blast each side separately
if split_nt is not None: 
    blast_res['LHS'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs_LHS.fa'),
                                     blastdb = blast_db,
                                     path2blastn=(blast_exec_path + 'blastn'),
                                     outfile = (out_dir + 'kmers_candidates_filtered_blast_output_LHS.txt'))
    blast_res['RHS'] = run_blast(fasta=(out_dir + 'kmers_candidates_filtered_transcript_seqs_RHS.fa'),
                                     blastdb = blast_db,
                                     path2blastn=(blast_exec_path + 'blastn'),
                                     outfile = (out_dir + 'kmers_candidates_filtered_blast_output_RHS.txt'))

In [ ]:
for k in blast_res.keys():
    print(("The following genes were detected in mode: " +  k))
    print(blast_res[k]['sgeneid'].value_counts().head(10))

In [ ]:
blast_res['full']

Not all BLAST hits will be off-targets. Hopefully, our gene of interest is included in the BLAST output. We therefore need to filter for hits with different gene IDs.

In [ ]:
offtargets = []
for k in blast_res.keys():
    offtargets += (detect_offtargets(blast_res[k], gene_ID, min_mismatches=min_mismatches))
## Remove redundancies
offtargets = list(set(offtargets))

In [ ]:
len(offtargets)

In [ ]:
## Remove off-targets
kmers = kmers[kmers['name'].isin(offtargets)==False].reset_index(drop=True)

In [ ]:
kmers 

### 6. Select non-overlapping probes

At this point, we have effectively acquired a set of usable probes. They don't overlap undesirable regions (repeats/polymorphism), have desirable sequence features (GC content, specific nucleotides) and are specific to our gene of interest.

In this particular case, we still have a lot of possible k-mers (much more than the number of probes we intend to design). We can therefore choose to prioritise k-mers with shorter homopolymer stretches, as these are also discouraged by the [10x recommendations](https://cdn.10xgenomics.com/image/upload/v1697739385/support-documents/CG000621_CustomProbeDesign_TechNote_RevC.pdf).

However, at this stage you might want to consider ranking probes in a diffferent way, depending on your application.

After having ranked our k-mers in whatever way we think is reasonable at this stage, we can proceed with selecting the top probe, then removing all overlapping/adjacent probes (within a window determined by `probe_offset`).

Since we have so many available probes, we will be increasing `probe_offset` from `100 (default)` to `1000`.

In [ ]:
## Sort in increasing homopolymer length
kmers = kmers.sort_values('longest_homopolymer', ascending=True).reset_index(drop=True)

In [ ]:
kmers

In [ ]:
## We have a lot of probes here - increasing the offset to 1000 bp to space them out
# probe_offset = 1000 # uncomment to make changes to the top-level arguments 

In [ ]:
# Select probes by row coordinates instead of a potentially duplicated name.
selected_probes_list = []
df = kmers.copy()

while len(selected_probes_list) < n_desired_probes and not df.empty:
    selected_probe = df.iloc[[0]].copy()
    selected_probes_list.append(selected_probe)

    start = int(selected_probe["start"].iloc[0])
    end = int(selected_probe["end"].iloc[0])

    df = (
        df.loc[
            (df["end"] < start - probe_offset)
            | (df["start"] > end + probe_offset)
        ]
        .reset_index(drop=True)
        .copy()
    )

selected_probes = pd.concat(selected_probes_list, ignore_index=True)

NameError: name 'kmers' is not defined

In [ ]:
selected_probes_df = pd.concat(selected_probes_list, axis=0).reset_index(drop=True)

In [ ]:
selected_probes_df

In [ ]:
for seq in selected_probes_df['transcript_seq']:
    print(seq)

And we are done! We have now selected three potential probes for our gene.

If our probes are meant to be split, we can additionally generate these columns:

In [ ]:
if split_nt is not None: 
    ## Make split probes (and add adapters if provided)
    selected_probes_df['probe_seq_LHS'] = [(LHS_pref + seq[0:split_nt] + LHS_suff) for seq in selected_probes_df['probe_seq']]
    selected_probes_df['probe_seq_RHS'] = [(RHS_pref +seq[split_nt: probe_length] + RHS_suff) for seq in selected_probes_df['probe_seq']]

In [ ]:
## Also add gene_ID for completeness
selected_probes_df['gene_ID'] = gene_ID

In [ ]:
## Export selected probes as dataframe:
selected_probes_df.to_csv((out_dir + 'kmers_selected_probes.csv'))

We always recommend additionally performing a manual BLAST of these probe sequences to make sure that there are no off-target effects.

In [ ]:
selected_probes_df